# Workforce Intelligence: Phase 4 Attrition Risk Modelling & Out-of-Sample Validation

This presentation notebook details the leakage-safe development, probability calibration, and out-of-sample holdout validation of attrition risk models developed on the IBM HR analytics dataset.

> [!IMPORTANT]
> **Decision Support Only:** The models developed here provide ranking and triage support for human HR professionals > and line managers. **NO automated adverse action** is permitted.

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display, Markdown

manifest = pd.read_csv('reports/phase4/figure_manifest.csv')
display(Markdown('### Phase 4 Figure Manifest'))
manifest[['figure_id', 'filename', 'title', 'plotted_data_sha256']]

## 1. Cross-Validation Model Development & Feature Set Governance

- **Training Split**: $n = 1,176$ (190 departures, 16.16% baseline).
- **Evaluation Protocol**: Repeated Stratified 5-Fold Cross-Validation (3 repeats = 15 folds).
- **Feature Set Governance**: Set B (demographics excluded) was selected because Set A (demographics included) did not exceed the pre-specified materiality threshold (+0.0300 PR-AUC gain).

In [ ]:
cv_metrics = pd.read_csv('reports/phase4/tables/cv_metrics.csv')
display(cv_metrics[['model', 'feature_set', 'cv_pr_auc_mean', 'cv_pr_auc_std', 'cv_roc_auc_mean', 'cv_brier_score_mean']])

## 2. Out-of-Sample Holdout Performance ($n = 294$, 47 Departures)

The holdout set was locked and scored exactly once at the conclusion of model development. Uncertainty intervals are 2,000-sample percentile bootstrap 95% confidence intervals.

In [ ]:
holdout_df = pd.read_csv('reports/phase4/tables/holdout_metrics.csv')
display(holdout_df[['model', 'pr_auc_point', 'pr_auc_ci95_low', 'pr_auc_ci95_high', 'recall_at_10_point', 'recall_at_20_point', 'brier_score_point']])

### FIG-15: Precision-Recall Curves across Attrition Risk Models

*Evaluates model precision across all recall thresholds on unseen holdout data. Calibrated Logistic Regression (Set B) achieves PR-AUC = 0.5690 vs chance baseline of 0.1599.*

In [ ]:
display(Image(filename='reports/phase4/figures/fig15_precision_recall_curves.png'))

### FIG-16: Probability Reliability Diagram (Calibration Curve)

*Demonstrates the effect of Platt/Sigmoid probability scaling. Calibrated probabilities align closely with empirical departure frequencies across all 5 risk bins.*

In [ ]:
display(Image(filename='reports/phase4/figures/fig16_calibration_curves.png'))

### FIG-17: Holdout Attrition Yield Across Prioritized Risk Tiers

*Partitions the holdout workforce into Top 10%, Next 10%, and Remaining 80%. The top 10% risk tier yields an observed departure rate of 65.5% (nearly 4x the baseline).*

In [ ]:
display(Image(filename='reports/phase4/figures/fig17_risk_tier_attrition.png'))

### FIG-18: Top Predictors in Regularised Logistic Regression

*Forest plot of standardized odds ratios with 95% confidence intervals. Overtime (OR = 2.00) and frequent business travel (OR = 1.73) dominate as operational risk factors.*

In [ ]:
display(Image(filename='reports/phase4/figures/fig18_feature_odds_ratios.png'))

### FIG-19: Out-of-Sample Validation of Candidate Attrition Hotspots

*Compares training split rates to holdout split rates for screened candidate segments. Identifies segments that generalized out-of-sample vs those that faded or were underpowered in holdout.*

In [ ]:
display(Image(filename='reports/phase4/figures/fig19_hotspot_validation.png'))

### FIG-20: Demographic Fairness Audit at Operational 20% Triage Threshold

*Audits model selection rate and leaver recall across Gender, MaritalStatus, and AgeGroup. Highlights disparate impact implications under the Four-Fifths rule in proactive support triage.*

In [ ]:
display(Image(filename='reports/phase4/figures/fig20_demographic_fairness.png'))

## 3. Hotspot Candidate Holdout Validation Summary Table

In [ ]:
val_table = pd.read_csv('reports/phase4/tables/hotspot_validation.csv')
display(val_table[['segment', 'train_rate', 'holdout_n', 'holdout_rate', 'holdout_ci_low', 'holdout_ci_high', 'validation_status']].head(15))

## 4. Demographic Fairness Audit Summary Table

In [ ]:
fair_table = pd.read_csv('reports/phase4/tables/demographic_fairness.csv')
display(fair_table[['attribute', 'subgroup', 'sample_size', 'selection_rate', 'recall', 'disparate_impact_ratio']])